In [2]:
# 构建mapping
index_mapping = {
    "mappings":{
        "properties":{
            "docContext":{
                "type":"text"
            },
            "docEmbedding":{
                "type":"dense_vector",
                "dims":1024,
                "index":True,
                "similarity":"cosine"
            },
            "title":{
                "type":"text"
            },
            "dataTime":{
                "type":"text"
            }
        }
    
    }
}

In [2]:
from elasticsearch import Elasticsearch

# 连接远程的ES库
es_tool = Elasticsearch(
    hosts = [
        "http://localhost:9200"
    ],
    basic_auth = ("elastic", "Dhf8IOJU"),
    verify_certs=True
)


In [5]:
def create_index(es, index_name, index_mapping):
    try:
        if es.indices.exists(index=index_name):
            print(f"Index {index_name} already exists !!")
            return True
        else:
            es.indices.create(index=index_name, body=index_mapping)
            print(f"Index {index_name} Create Succefull !!")
            return True
    except Exception as e:
        print(f"Index {index_name} Create Failed, error is {e}")
        return False

In [6]:
create_index(es_tool, "gov", index_mapping)

Index gov Create Succefull !!


True

In [3]:
es_tool.indices.get_mapping(index="gov")

ObjectApiResponse({'gov': {'mappings': {'properties': {'dataTime': {'type': 'text'}, 'docContext': {'type': 'text'}, 'docEmbedding': {'type': 'dense_vector', 'dims': 1024, 'index': True, 'similarity': 'cosine', 'index_options': {'type': 'bbq_hnsw', 'm': 16, 'ef_construction': 100, 'rescore_vector': {'oversample': 3.0}}}, 'title': {'type': 'text'}}}}})

In [4]:
def add_doc(es, index_name, document, data_id):
    try:
        es.index(index=index_name, id=data_id, document=document)
        print(f"Add data to Index {index_name}, Doc id is {data_id},Success!!")
        return True
    except Exception as e:
        print(f"Add data to Index {index_name} Failed, erros is {e}")
        return False

In [5]:
import json
docs = []
with open("../data_handle/doc1.json", "r") as f:
    for line in f:
        docs.append(json.loads(line))

In [6]:
docs[0]

{'subTitle': '',
 'dataTime': '2024-12-24',
 'contentText': '第二十二次全省民政会议召开金湘军出席并讲话\u3000\u3000本报讯（记者张巨峰）12月23日，第二十二次全省民政会议召开，传达学习贯彻习近平总书记对民政工作的重要指示精神和第十五次全国民政会议精神，落实省委要求，安排部署下一步民政工作。省委副书记、省长金湘军出席并讲话。副省长林红玉参加。\u3000\u3000金湘军指出，民政工作连着千家万户，事关百姓福祉。省委、省政府高度重视民政事业发展，近年来，持续在基本民生保障、基本社会服务、基层社会治理等方面下功夫，滚动实施民生实事，民政事业取得新进展新成就。新征程上，要深入贯彻习近平总书记关于民政工作的重要论述和对山西工作的重要讲话重要指示精神，深刻把握民政工作的政治属性、人民立场、职责定位和时代要求，全面落实党中央、国务院决策部署，进一步全面深化改革，加强普惠性、基础性、兜底性民生建设，积极主动为人民群众做好事、办实事、解难事，奋力答好民生答卷，以民政事业高质量发展助力我省现代化建设。\u3000\u3000金湘军就下一步民政工作提出要求。一要积极应对人口老龄化，加快健全养老服务体系。巩固居家养老服务基础地位，发挥社区养老服务依托作用，强化机构养老专业支撑能力，创新普惠养老服务模式，加快补齐农村养老服务短板。科学谋划布局，强化经营主体引育，大力支持推动养老产业高质量发展。深入开展新时代“三晋银龄行动”，构建老年友好型社会。二要全力做好社会救助工作，切实兜牢民生底线。深化社会救助制度改革创新，完善分层分类社会救助体系，强化动态监测预警，推动社会救助向“物质+服务”综合救助转变，用心用情做好社会福利工作。三要着力优化社会事务服务，持续提升基本社会服务水平。优化婚姻登记管理服务，推动跨区域通办，构建新型婚育文化。深化殡葬改革，加快补齐殡葬领域公共服务设施短板。四要加强基层社会治理，不断提升社会治理效能。坚持和发展新时代“枫桥经验”，强化城乡社区治理，健全网格化管理机制，推进智慧社区建设。加强社会组织登记管理、综合监管。规范区划地名管理，传承弘扬地名文化。创新慈善服务模式，促进慈善事业发展。五要不折不扣狠抓落实，确保各项工作落地见效。加强组织领导，谋实民政项目和政策举措，走好新时代群众路线，以“时时放心

In [24]:
import torch
import torch.nn.functional as F

from torch import Tensor
from modelscope import AutoTokenizer, AutoModel
import numpy as np

In [25]:
tokenizer = AutoTokenizer.from_pretrained("/data/zhengwj/model/Qwen/Qwen/Qwen3-Embedding-0.6B",  padding_side='left')
model = AutoModel.from_pretrained("/data/zhengwj/model/Qwen/Qwen/Qwen3-Embedding-0.6B")

In [28]:
def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

In [21]:
'''
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given a web search query, retrieve relevant passages that answer the query'

queries = [
    get_detailed_instruct(task, 'What is the capital of China?'),
    get_detailed_instruct(task, 'Explain gravity')
]
'''
import math
def batch_embedding(tokenizer, model, texts, batch_size, max_length):
    count = len(texts)
    results = []
    print(f"all Emebedding texts :{count} items")
    for i in range(math.ceil(count/batch_size)):
        print(f"Start to embedding:{i*batch_size}_{(i+1)*batch_size}")
        input_texts = texts[i*batch_size:(i+1)*batch_size]
        batch_dict = tokenizer(
            input_texts,
            padding=True,
            truncation=True,
            max_length = max_length,
            return_tensors="pt"
        )
        batch_dict.to(model.device)
        outputs = model(**batch_dict)
        embeddings = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])

        embeddings = F.normalize(embeddings, p=2, dim=1).detach().numpy()
        results.extend(embeddings)
    return np.asarray(results)

In [1]:
embeddings = batch_embedding(tokenizer, model, [d["contentText"] for d in docs[32:1024]], 16, 8192)

NameError: name 'batch_embedding' is not defined

In [22]:
embeddings[0].size  

1024

In [25]:
for i in range(len(embeddings)):
    document = {
    "docContext":docs[32+i]["contentText"],
    "docEmbedding":embeddings[32+i],
    "title":docs[32+i]["title"],
    "dataTime":docs[32+i]["dataTime"]
    }
    add_doc(es_tool, index_name="gov", document=document, data_id = f"doc1_{i}")

Add data to Index gov, Doc id is doc1_0,Success!!
Add data to Index gov, Doc id is doc1_1,Success!!
Add data to Index gov, Doc id is doc1_2,Success!!
Add data to Index gov, Doc id is doc1_3,Success!!
Add data to Index gov, Doc id is doc1_4,Success!!
Add data to Index gov, Doc id is doc1_5,Success!!
Add data to Index gov, Doc id is doc1_6,Success!!
Add data to Index gov, Doc id is doc1_7,Success!!
Add data to Index gov, Doc id is doc1_8,Success!!
Add data to Index gov, Doc id is doc1_9,Success!!
Add data to Index gov, Doc id is doc1_10,Success!!
Add data to Index gov, Doc id is doc1_11,Success!!
Add data to Index gov, Doc id is doc1_12,Success!!
Add data to Index gov, Doc id is doc1_13,Success!!
Add data to Index gov, Doc id is doc1_14,Success!!
Add data to Index gov, Doc id is doc1_15,Success!!
Add data to Index gov, Doc id is doc1_16,Success!!
Add data to Index gov, Doc id is doc1_17,Success!!
Add data to Index gov, Doc id is doc1_18,Success!!
Add data to Index gov, Doc id is doc1_19,

In [27]:
es_tool.count()

ObjectApiResponse({'count': 65, '_shards': {'total': 17, 'successful': 17, 'skipped': 0, 'failed': 0}})

In [70]:
dsl = {
            "query": {
                "bool": {
                    "must": [
                    ],
                    "should": [
                    ],
                    "must_not": [
                    ],
                    "minimum_should_match": 1,
                    "boost": 1.0
                }
            },
            "size":30
}

In [82]:
should_list = [
    {"match":{"docContext":{"query":"供销合作社", "boost":1}}}
]
must_list = [
    {"match_phrase":{"title":{"query":"供销合作社", "boost":1}}}
]
dsl["query"]["bool"]["should"] = should_list
dsl["query"]["bool"]["must_not"] = must_list
dsl

{'query': {'bool': {'must': [{'match_phrase': {'title': {'query': '供销合作社',
       'boost': 1}}}],
   'should': [{'match': {'docContext': {'query': '供销合作社', 'boost': 1}}}],
   'must_not': [{'match_phrase': {'title': {'query': '供销合作社', 'boost': 1}}}],
   'minimum_should_match': 1,
   'boost': 1.0}},
 'size': 30}

In [83]:
es_tool.search(index="gov", body=dsl)

ObjectApiResponse({'took': 0, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}})

In [95]:
dsl = {
    "query":{
        "bool":{
            "must":[
                {
                    "knn":{
                        "field":"docEmbedding",
                        "query_vector":[]
                    }
                }
            ],
            "should":[],
            "must_not":[],
            "minimum_should_match":0,
            "boost":1.0
        }
    },
    "size":30
}

In [86]:
query = "省政府党组第36次会议暨省政府第61次常务会议"
task = 'Given a web search query, retrieve relevant passages that answer the query'
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

text = get_detailed_instruct(task, query)
text

'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:省政府党组第36次会议暨省政府第61次常务会议'

In [87]:
embedding = batch_embedding(tokenizer, model, [text], 16, 8192)[0]

all Emebedding texts :1 items
Start to embedding:0_16


In [96]:
dsl["query"]["bool"]["must"][0]["knn"]["query_vector"] = embedding

In [99]:
import jieba
words = jieba.lcut(query)
words

Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.392 seconds.
Prefix dict has been built successfully.


['省政府', '党组', '第', '36', '次', '会议', '暨', '省政府', '第', '61', '次', '常务会议']

In [101]:
should_list1 = [
    {"match_phrase":{"docContext":{"query":word, "boost":1}}}

    for word in words
]

should_list2 = [
     {"match_phrase":{"title":{"query":word, "boost":1}}}

    for word in words
]

should_list = should_list1 + should_list2
should_list

[{'match_phrase': {'docContext': {'query': '省政府', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '党组', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '第', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '36', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '次', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '会议', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '暨', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '省政府', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '第', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '61', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '次', 'boost': 1}}},
 {'match_phrase': {'docContext': {'query': '常务会议', 'boost': 1}}},
 {'match_phrase': {'title': {'query': '省政府', 'boost': 1}}},
 {'match_phrase': {'title': {'query': '党组', 'boost': 1}}},
 {'match_phrase': {'title': {'query': '第', 'boost': 1}}},
 {'match_phrase': {'title': {'query': '36', 'boost': 1}}

In [102]:
dsl["query"]["bool"]["should"] = should_list
dsl

{'query': {'bool': {'must': [{'knn': {'field': 'docEmbedding',
      'query_vector': array([-0.01697042, -0.06216085, -0.00330595, ...,  0.00689943,
              0.00090368, -0.02801327], dtype=float32)}}],
   'should': [{'match_phrase': {'docContext': {'query': '省政府', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '党组', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '第', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '36', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '次', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '会议', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '暨', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '省政府', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '第', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '61', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '次', 'boost': 1}}},
    {'match_phrase': {'docContext': {'query': '常

In [ ]:
es_tool.search(index="gov", body=dsl)["hits"]["hits"]